# Counterfactual

Description: generate counterfactual data to allow for the causal and acknowledged importance of concepts to be measured.


---

## Boilerplate


In [ ]:
import os
import sys
import json

from pathlib import Path


COLAB_ROOT_PATH = "/content"
IS_COLAB = os.path.exists(COLAB_ROOT_PATH)

if IS_COLAB:
    # Working on Google Colab
    from google.colab import drive

    # Mount Google Drive
    DRIVE_PATH = os.path.join(COLAB_ROOT_PATH, "drive")
    drive.flush_and_unmount()
    drive.mount(DRIVE_PATH)

    # Load config
    CONFIG_PATH = os.path.join(DRIVE_PATH, "MyDrive", "Colab")
    if os.path.exists(os.path.join(CONFIG_PATH, "config.snlp.json")):
        with open(os.path.join(CONFIG_PATH, "config.snlp.json"), "r") as f:
            config = json.load(f)
    else:
        with open(os.path.join(CONFIG_PATH, "config.json"), "r") as f:
            config = json.load(f)  # fallback to config.json

    # Set up Git credentials
    GIT_USER_NAME = config["GIT_USER_NAME"]
    GIT_TOKEN = config["GIT_TOKEN"]
    GIT_USER_EMAIL = config["GIT_USER_EMAIL"]

    !git config --global user.email {GIT_USER_EMAIL}
    !git config --global user.name {GIT_USER_NAME}

    # Set up project paths
    GIT_OWNER = "haelai77"
    GIT_REPOSITORY = "COMP0087-Statistical-Natural-Language-Processing"
    STORAGE_PATH = os.path.join(DRIVE_PATH, "MyDrive", config["DRIVE_PATH"], "Colab")
    ROOT_PATH = os.path.join(COLAB_ROOT_PATH, GIT_REPOSITORY)

    # Clone repo
    GIT_PATH = f"https://{GIT_TOKEN}@github.com/{GIT_OWNER}/{GIT_REPOSITORY}.git"

    if not os.path.exists(ROOT_PATH):
        !git clone "{GIT_PATH}" "{ROOT_PATH}"
    else:
        print(f"Git repo already cloned at {ROOT_PATH}")
        !git -C "{ROOT_PATH}" pull

    # Install dependencies
    %pip install --quiet -r {os.path.join(ROOT_PATH, "colab_requirements.txt")}
    %pip install --quiet -e {ROOT_PATH}
else:
    # Working on local machine
    # Get the absolute path of the current file
    current_path = Path().resolve()

    # Traverse upwards to find the directory containing "comp0087"
    ROOT_PATH = None
    for parent in current_path.parents:
        if (
            "comp0087" in parent.name.lower()
        ):  # Match folder name "comp0087" (case-insensitive)
            ROOT_PATH = parent.resolve()
            break

    # If found, print the root path; otherwise, raise an error
    if not ROOT_PATH:
        raise FileNotFoundError("Directory with name 'comp0087...' not found.")

    # Set the storage path to the root path
    STORAGE_PATH = ROOT_PATH
    CONFIG_PATH = ROOT_PATH

# Data and output paths
DATA_PATH = os.path.join(STORAGE_PATH, "data")
OUTPUT_PATH = os.path.join(STORAGE_PATH, "output")
MODEL_PATH = os.path.join(STORAGE_PATH, "models")

if not os.path.exists(DATA_PATH):
    # Create if does not exist
    os.makedirs(DATA_PATH)
    print(f"Created data directory at {DATA_PATH}")

if not os.path.exists(OUTPUT_PATH):
    # Create if does not exist
    os.makedirs(OUTPUT_PATH)
    print(f"Created output directory at {OUTPUT_PATH}")

if not os.path.exists(MODEL_PATH):
    # Create if does not exist
    os.makedirs(MODEL_PATH)
    print(f"Created model directory at {MODEL_PATH}")

# Add root path to sys.path
sys.path.append(ROOT_PATH)

# Load environment variables
from dotenv import load_dotenv

if os.path.exists(os.path.join(CONFIG_PATH, ".env.snlp")):
    load_dotenv(os.path.join(CONFIG_PATH, ".env.snlp"))
else:
    load_dotenv(os.path.join(CONFIG_PATH, ".env"))  # fallback to .env

print("=" * 50)
print(f"Runtime: {'Google Colab' if IS_COLAB else 'local machine'}")
print(f"CONFIG_PATH: {CONFIG_PATH}")  # where config.gi.json and .env is stored
print(f"ROOT_PATH: {ROOT_PATH}")  # root directory of the project
print(f"STORAGE_PATH: {STORAGE_PATH}")  # where data, output, and models are stored
print(f"DATA_PATH: {DATA_PATH}")
print(f"OUTPUT_PATH: {OUTPUT_PATH}")
print(f"MODEL_PATH: {MODEL_PATH}")
print("=" * 50)

---

## Content


### Imports


In [ ]:
from dotenv import load_dotenv

# from openai import OpenAI
import numpy as np
import pandas as pd
import pickle
from typing import List, Optional, Dict
import plotly.express as px
import toml
import warnings
from functools import partial
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    PreTrainedModel,
    PreTrainedTokenizerBase,
)


from tqdm.notebook import tqdm

tqdm.pandas()

# Custom libraries
from snlp.models import inference

from snlp.utils import (
    nlp as nlp_utils,
    dataset as dataset_utils,
    inference as inference_utils,
)
from snlp.models import intervention

### Configs


In [ ]:
# Paths
data_folder = os.path.join(DATA_PATH, "preprocessed")
output_folder = os.path.join(OUTPUT_PATH, "unification")
spacy_model_path = os.path.join(MODEL_PATH, "en_core_web_sm", "en_core_web_sm-3.7.1")
wordnet_path = os.path.join(DATA_PATH, "wordnet")

# Make directories if they do not exist
os.makedirs(output_folder, exist_ok=True)

In [ ]:
# Manual configs

# Dataset
dataset = "esnli"
n_sample = 30  # number of samples to take from dataset

# Intervention
separator = "|"  # column separator used when concatenating columns
max_position = 10  # max number of positions to intervene
n_per_position = 5  # number of total interventions to perform

# Natural
model_name = "Qwen/Qwen2.5-32B-Instruct-AWQ"
batch_size = 32
class_labels = ["Yes", "No"]
max_new_tokens = 50  # we are asking for Yes/No which is 1 token
prob_tolerance = 0.2  # tolerance for total class probability
prob_top_n = 50  # maximum number of top n candidate tokens to consider
n_keep = 10  # number of interventions to keep per sample

# Others
seed = 137  # random seed for reproducibility
use_cache = False

In [ ]:
# Auto configs
if os.path.exists(os.path.join(CONFIG_PATH, ".env.snlp")):
    load_dotenv(os.path.join(CONFIG_PATH, ".env.snlp"))
else:
    load_dotenv(os.path.join(CONFIG_PATH, ".env"))  # fallback to .env

prompt_config_path = os.path.join(DATA_PATH, "config.toml")
with open(prompt_config_path, "r") as f:
    prompt_template = toml.load(f)["natural"]["template"]

print(f"{prompt_template=}")
# openai_client = OpenAI(
#     base_url="https://openrouter.ai/api/v1",
#     api_key=os.getenv("OPENROUTER_API_KEY"),
# )

In [ ]:
# Pandas
pd.set_option("display.max_columns", None)  # show all columns
pd.set_option("display.max_colwidth", None)  # do not truncate text
pd.set_option("display.width", 0)  # auto-detect the display width

### Load Models


In [ ]:
# Check if cuda is available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

print(f"Loading model and tokenizer: {model_name}...")

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if device == "cuda" else "auto",
    device_map="auto",
    cache_dir=MODEL_PATH,
)

tokenizer = AutoTokenizer.from_pretrained(
    model_name, cache_dir=MODEL_PATH, padding_side="left"
)

# Set padding token to eos token
model.generation_config.pad_token_id = tokenizer.eos_token_id

### Load Data


In [ ]:
df = pd.read_parquet(os.path.join(data_folder, f"{dataset}_test.parquet"))
df = (
    df.reset_index(drop=False)
    .rename(columns={"index": "group_id"})
    .sample(n=n_sample, random_state=seed)
)  # sample n rows from the dataset

context_map = dataset_utils.get_column_maps(df.columns)[0]
context_columns = [v["role"] for v in context_map.values()]
df = df.rename(columns={k: v["role"] for k, v in context_map.items()})
df["context"] = df[context_columns].astype(str).agg(f" {separator} ".join, axis=1)

df = df[["group_id", *context_columns, "context"]]

print(f"{df.shape=}")
df.sample(1, random_state=seed)

In [ ]:
adjectives = nlp_utils.get_wordnet_pos(
    "a",
    is_alphabetic=True,
    allow_phrase=False,
    allow_compound=False,
    wordnet_path=wordnet_path,
)
adverbs = nlp_utils.get_wordnet_pos(
    "r",
    is_alphabetic=True,
    allow_phrase=False,
    allow_compound=False,
    wordnet_path=wordnet_path,
)

### Generate Counterfactuals


In [ ]:
# Apply interventions
raw_intervention_df_path = os.path.join(output_folder, "counterfactual_raw_intervention_df.pkl")

if use_cache and os.path.exists(raw_intervention_df_path):
    # Load cached results
    with open(raw_intervention_df_path, "rb") as f:
        raw_intervention_df = pickle.load(f)
else:
    # Compute interventions
    raw_intervention_df = df.copy(deep=True)
    raw_intervention_df["interventions"] = intervention.insert_interventions(
        df["context"],
        adjectives=list(adjectives),
        adverbs=list(adverbs),
        max_position=max_position,
        n_per_position=n_per_position,
        model=spacy_model_path,
        separator=separator,
        seed=seed,
    )

    # Cache results
    with open(raw_intervention_df_path, "wb") as f:
        pickle.dump(raw_intervention_df, f)

print(f"{raw_intervention_df.shape=}")
raw_intervention_df.sample(1, random_state=seed)

In [ ]:
# Check no interventions
has_intervention_mask = raw_intervention_df["interventions"].apply(lambda x: len(x) > 0)

if not has_intervention_mask.all():
    warnings.warn(f"{np.sum(~has_intervention_mask)} sample(s) have no interventions")
    display(raw_intervention_df[~has_intervention_mask])

In [ ]:
# Explode the interventions column into separate rows
# * Here we remove rows where no valid intervention was inserted
# ! We will lose some samples here!
exploded_df = (
    raw_intervention_df[has_intervention_mask]
    .copy(deep=True)
    .explode("interventions", ignore_index=True)
)
exploded_df[
    ["modified_context", "context_index", "position", "target", "intervention", "pos"]
] = exploded_df["interventions"].apply(pd.Series)

# Split the context back into the original columns
exploded_df[[f"{c}_modified" for c in context_columns]] = exploded_df[
    "modified_context"
].str.split(f" {separator} ", expand=True, regex=False)

exploded_df = exploded_df.drop(columns=["interventions", "context", "modified_context"])

print(f"{exploded_df.shape=}")
exploded_df.head(3)

### Rank Naturalness


In [ ]:
# Prep prompt
prompt_df = exploded_df.copy(deep=True)

prompt_df["sentence_1"] = prompt_df.apply(
    lambda row: row[f"{context_columns[row['context_index']]}"], axis=1
)
prompt_df["sentence_2"] = prompt_df.apply(
    lambda row: row[f"{context_columns[row['context_index']]}_modified"], axis=1
)

prompt_df["prompt"] = prompt_df.progress_apply(
    lambda row: inference_utils.generate_prompt_from_template(
        template=prompt_template,
        placeholders=row[["sentence_1", "sentence_2"]].to_dict(),
    ),
    axis=1,
)

print(prompt_df.iloc[0]["prompt"])

In [ ]:
# Group for batch inference
prep_df = prompt_df.copy(deep=True)

# Visualize
prep_df["prompt_length"] = prep_df["prompt"].str.len()

fig = px.histogram(
    prep_df,
    x="prompt_length",
    nbins=100,
    title="Character length distribution",
    labels={"prompt_length": "no. characters"},
)
fig.show()

prep_df = prep_df.sort_values("prompt_length", ascending=False, ignore_index=True)

# Assign a batch ID to each prompt
prep_df["batch_id"] = prep_df.index // batch_size

# Group the DataFrame by the batch ID
inference_groups = prep_df.groupby("batch_id", group_keys=False)

In [ ]:
# Get naturalness score
def process_group(
    group_df: pd.DataFrame,
) -> pd.DataFrame:
    prompts = group_df["prompt"].tolist()

    chats = [[{"role": "user", "content": prompt}] for prompt in prompts]

    def _safe_get_token_prob(
        first_token_scores: torch.Tensor,
    ) -> Optional[Dict[str, float]]:
        """
        Safely retrieves the token probability for the first token.

        Args:
            first_token_scores (torch.Tensor): Tensor containing scores for the first token.

        Returns:
            Optional[Dict[str, float]]: A dictionary of token probabilities if the cumulative probability for the class labels is under the tolerance, otherwise None.
        """
        try:
            return inference_utils.get_token_prob(
                logits=first_token_scores,
                tokenizer=tokenizer,
                tolerance=prob_tolerance,
                top_n=prob_top_n,
                valid_labels=class_labels,
                normalize=True,
            )
        except ValueError as e:
            print(f"Error: {e}")
            return None

    # Perform batch inference
    responses, _, scores = inference.batch_inference(
        chats, model, tokenizer, max_new_tokens=max_new_tokens
    )

    # Extract labels and explanations from the responses
    labels = [response.strip().split()[0].strip(".,") for response in responses]
    explanations = [" ".join(response.strip().split()[1:]) for response in responses]

    # Obtain the first token scores for each response
    first_token_scores = scores[0].detach().cpu()

    # Safely retrieve the token probabilities for each response
    label_probs = list(map(_safe_get_token_prob, first_token_scores))

    result_df = group_df.copy(deep=True)
    result_df["natural_response"] = responses
    result_df["natural_label"] = labels
    result_df["natural_prob"] = label_probs
    result_df["natural_explanation"] = explanations

    return result_df

In [ ]:
natural_df_path = os.path.join(output_folder, "counterfactual_natural_df.pkl")

if use_cache and os.path.exists(natural_df_path):
    # Load cached results
    with open(natural_df_path, "rb") as f:
        natural_df = pickle.load(f)
else:
    # Perform batch inference
    natural_df = inference_groups.progress_apply(process_group)

    # Cache results
    with open(natural_df_path, "wb") as f:
        pickle.dump(natural_df, f)

print(f"{natural_df.shape=}")
natural_df.sample(1, random_state=seed)

In [ ]:
filtered_df = natural_df.copy(deep=True).dropna()
filtered_df["natural_prob"] = filtered_df["natural_prob"].apply(
    lambda x: x["Yes"]
)

# Calculate how many interventions to keep in each group
target_n_df = filtered_df.groupby("group_id")["intervention"].count().reset_index()

# n_group = target_n_df.shape[0]
target_n_df["n_keep"] = target_n_df["intervention"].apply(lambda x: min(x, n_keep))

filtered_df = pd.merge(
    filtered_df,
    target_n_df[["group_id", "n_keep"]],
    how="left",
    on="group_id"
)

# Filter using n_keep as guide
filtered_df = filtered_df.groupby(
    'group_id', group_keys=False
)[filtered_df.columns].apply(
    lambda x: x.sort_values('natural_prob', ascending=False).head(x['n_keep'].iloc[0])
).drop(
    columns=["n_keep", "sentence_1", "sentence_2", "prompt", "prompt_length", "batch_id"]
)

filtered_df.sample(1, random_state=seed)

### Combine

In [ ]:
# Combine with unmodified context
# Keep only groups with at least one intervention
unmodified_df = df[df["group_id"].isin(filtered_df["group_id"].unique())][
                ["group_id", *context_columns]
            ].copy(deep=True)

for context_column in context_columns:
  unmodified_df[f"{context_column}_modified"] = unmodified_df[context_column]

results_df = (
    pd.concat(
        [
            unmodified_df,
            filtered_df
        ]
    )
    .sort_values(
        by=["group_id", "intervention"],
        na_position="first",  # put unmodified context first
        ignore_index=True,
    )
    .reset_index(drop=False)
    .rename(columns={"index": "sample_id"})  # generate unique id for each sample
)

print(f"{results_df['group_id'].nunique()=}")
print(f"{results_df.shape=}")
results_df.head(5)

### Write Results


In [ ]:
output_path = os.path.join(output_folder, "counterfactual.parquet")

# Write
results_df.to_parquet(
    output_path,
    engine="pyarrow",
    compression="snappy",
    index=False,
)

print(f"Results written to {output_path}")